# qlib工作流
这份 notebook 保持 `workflow_by_code.ipynb` 的显式思路，但底层数据准备、滚动评估和回测分析复用 ValueInvesting 仓库现有的周频研究链路。

## 1. 目标与官方 Workflow 映射

本 notebook 的主线是：Universe 解析与 panel 导出 -> 特征准备 -> 显式训练 -> 显式预测 -> 显式滚动回测 -> 分析与可视化 -> 可选发布。

与 qlib 官方 `workflow_by_code.ipynb` 的对应关系：

- `Data Prepare` -> 本仓库的 `export_or_load_panel()` 与周频特征 panel
- `Dataset / Model` -> `StaticDataLoader` + `DataHandlerLP` + `DatasetH` + `LGBModel`
- `Predict / Record` -> 显式 `model.predict()` 与预测表整理
- `Backtest / Analysis` -> 本仓库 `weekly_model_eval.py` 中的滚动评估和周频回测函数
- `Artifact Publish` -> `publish_score_snapshot()` 发布快照给线上智能选股使用

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from qlib_research.core.portfolio import BacktestTradingConfig
from qlib_research.core.notebook_workflow import build_segments, ensure_runtime, export_or_load_panel
from qlib_research.core.qlib_pipeline import (
    FEATURE_GROUP_COLUMNS,
    LABEL_COLUMN,
    apply_industry_normalization,
    build_training_frame,
    compose_feature_columns,
    init_qlib_runtime,
    publish_score_snapshot,
    suppress_external_output,
)
from qlib_research.core.weekly_model_eval import (
    SLICE_FLAG_COLUMNS,
    attach_future_labels_to_prediction,
    build_backtest_price_frames,
    build_signal_matrix,
    default_weekly_net_backtest_config,
    evaluate_prediction_frame,
    fit_predict_one_date,
    prefilter_feature_columns,
    run_strategy_backtest,
    select_evaluation_dates,
    summarize_details,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

MONTH_HEATMAP_COLUMNS = [f"{month:02d}" for month in range(1, 13)]

display(Markdown(f"**PROJECT_ROOT**: `{PROJECT_ROOT}`"))


def _format_metric(value: object, digits: int = 4) -> str:
    if value is None or pd.isna(value):
        return "NA"
    return f"{float(value):,.{digits}f}"


def _panel_path_for_universe(universe_profile: str) -> str:
    mapping = {
        "watchlist": "artifacts/panels/watchlist_weekly.csv",
        "csi300": "artifacts/panels/csi300_weekly.csv",
        "csi500": "artifacts/panels/csi500_weekly.csv",
        "merged_csi300_500": "artifacts/panels/csi300500_weekly.csv",
    }
    return mapping[universe_profile]


def _output_dir_for_universe(universe_profile: str, date_tag: str) -> str:
    alias = "csi300500" if universe_profile == "merged_csi300_500" else universe_profile
    return f"artifacts/evaluations/{alias}_workflow_by_code_{date_tag}"


def _build_backtest_metric_frame(evaluation_name: str, metrics: dict[str, object]) -> pd.DataFrame:
    if not metrics:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "evaluation_name": evaluation_name,
                "strategy_total_return": metrics.get("strategy_total_return"),
                "strategy_cagr": metrics.get("strategy_cagr"),
                "strategy_max_drawdown": metrics.get("strategy_max_drawdown"),
                "strategy_weekly_sharpe": metrics.get("strategy_weekly_sharpe"),
                "strategy_total_cost": metrics.get("strategy_total_cost", 0.0),
                "strategy_commission": metrics.get("strategy_commission", 0.0),
                "strategy_exchange_fee": metrics.get("strategy_exchange_fee", 0.0),
                "strategy_transfer_fee": metrics.get("strategy_transfer_fee", 0.0),
                "strategy_stamp_duty": metrics.get("strategy_stamp_duty", 0.0),
                "strategy_impact_cost": metrics.get("strategy_impact_cost", 0.0),
                "strategy_gross_trade_value": metrics.get("strategy_gross_trade_value", 0.0),
                "strategy_turnover_ratio_total": metrics.get("strategy_turnover_ratio_total", 0.0),
                "strategy_trade_count": metrics.get("strategy_trade_count", 0.0),
                "backtest_mode": metrics.get("backtest_mode", "weekly_net"),
                "execution_price": metrics.get("execution_price", "next_open"),
            }
        ]
    )


def _build_monthly_return_heatmap_frame(equity_frame: pd.DataFrame) -> pd.DataFrame:
    if equity_frame.empty or "datetime" not in equity_frame.columns or "total_value" not in equity_frame.columns:
        return pd.DataFrame()
    frame = equity_frame[["datetime", "total_value"]].copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    frame["total_value"] = pd.to_numeric(frame["total_value"], errors="coerce")
    frame = frame.dropna(subset=["datetime", "total_value"]).sort_values("datetime")
    if frame.empty:
        return pd.DataFrame()

    frame["year"] = frame["datetime"].dt.year
    frame["month"] = frame["datetime"].dt.month
    monthly = (
        frame.groupby(["year", "month"], as_index=False)["total_value"]
        .agg(period_start="first", period_end="last")
    )
    monthly["return"] = (monthly["period_end"] / monthly["period_start"]) - 1.0
    heatmap = monthly.pivot(index="year", columns="month", values="return").sort_index()
    if heatmap.empty:
        return pd.DataFrame()
    heatmap = heatmap.reindex(columns=range(1, 13))
    heatmap.columns = MONTH_HEATMAP_COLUMNS
    return heatmap


def _build_annual_return_heatmap_frame(equity_frame: pd.DataFrame) -> pd.DataFrame:
    if equity_frame.empty or "datetime" not in equity_frame.columns or "total_value" not in equity_frame.columns:
        return pd.DataFrame()
    frame = equity_frame[["datetime", "total_value"]].copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    frame["total_value"] = pd.to_numeric(frame["total_value"], errors="coerce")
    frame = frame.dropna(subset=["datetime", "total_value"]).sort_values("datetime")
    if frame.empty:
        return pd.DataFrame()

    frame["year"] = frame["datetime"].dt.year.astype(str)
    annual = (
        frame.groupby("year", as_index=False)["total_value"]
        .agg(period_start="first", period_end="last")
    )
    annual["return"] = (annual["period_end"] / annual["period_start"]) - 1.0
    if annual.empty:
        return pd.DataFrame()
    return pd.DataFrame([annual["return"].tolist()], index=["annual_return"], columns=annual["year"].tolist())


def _render_return_heatmap(heatmap_frame: pd.DataFrame, title: str, x_label: str, y_label: str) -> None:
    if heatmap_frame.empty:
        display(Markdown(f"**{title}**：样本不足，无法生成热力图。"))
        return
    numeric_frame = heatmap_frame.apply(pd.to_numeric, errors="coerce") * 100.0
    finite_values = numeric_frame.to_numpy().ravel()
    finite_values = finite_values[pd.notna(finite_values)]
    symmetric_bound = max(abs(float(finite_values.min())), abs(float(finite_values.max())), 1.0) if len(finite_values) else 1.0
    fig = px.imshow(
        numeric_frame,
        text_auto=".1f",
        aspect="auto",
        color_continuous_scale=[
            [0.0, "#b2182b"],
            [0.5, "#f7f7f7"],
            [1.0, "#1a9850"],
        ],
        zmin=-symmetric_bound,
        zmax=symmetric_bound,
        origin="lower",
        title=title,
        labels={"x": x_label, "y": y_label, "color": "Return %"},
    )
    fig.update_layout(height=420)
    fig.show()


official_mapping_frame = pd.DataFrame(
    [
        ("Data Prepare", "Universe 解析 + panel 导出", "export_or_load_panel"),
        ("Feature Prepare", "特征组选择 + 预过滤 + 行业内标准化", "compose_feature_columns / prefilter_feature_columns / apply_industry_normalization"),
        ("Train", "显式 qlib 训练对象", "StaticDataLoader / DataHandlerLP / DatasetH / LGBModel"),
        ("Predict", "显式预测与分数整理", "model.predict / attach_future_labels_to_prediction"),
        ("Backtest", "周频滚动评估与组合净值", "fit_predict_one_date / build_signal_matrix / run_strategy_backtest"),
        ("Publish", "生成线上快照", "publish_score_snapshot"),
    ],
    columns=["official_node", "project_mapping", "key_api"],
)
display(official_mapping_frame)


**PROJECT_ROOT**: `/Volumes/Repository/Projects/TradingNexus/QlibResearch`

,official_node,project_mapping,key_api
0,Data Prepare,Universe 解析 + panel 导出,export_or_load_panel
1,Feature Prepare,特征组选择 + 预过滤 + 行业内标准化,compose_feature_columns / prefilter_feature_co...
2,Train,显式 qlib 训练对象,StaticDataLoader / DataHandlerLP / DatasetH / ...
3,Predict,显式预测与分数整理,model.predict / attach_future_labels_to_predic...
4,Backtest,周频滚动评估与组合净值,fit_predict_one_date / build_signal_matrix / r...
5,Publish,生成线上快照,publish_score_snapshot


## 2. 环境检查与运行时初始化

先确认 notebook 所需依赖是否可用。真正的 qlib runtime 会在第 6 节用 `init_qlib_runtime()` 显式初始化。

In [2]:
runtime_info = ensure_runtime()
runtime_frame = pd.DataFrame([runtime_info]).T.rename(columns={0: "value"})
display(runtime_frame)

assert runtime_info["pyqlib_version"], "未检测到 pyqlib，请先安装 qlib 依赖后再执行后续单元。"
assert runtime_info["finance_data_hub_version"], "未检测到 finance-data-hub，请先确认本地依赖和 sources.yml。"

display(Markdown("**说明**：qlib 运行时会在第 6 节显式调用 `init_qlib_runtime()` 初始化。"))

,value
project_root,/Volumes/Repository/Projects/TradingNexus/Valu...
cwd,/Volumes/Repository/Projects/TradingNexus/Valu...
python_version,3.12.10
artifacts_dir,/Volumes/Repository/Projects/TradingNexus/Valu...
ipykernel_version,7.2.0
plotly_version,6.5.1
finance_data_hub_version,0.1.0
pyqlib_version,0.9.7
qlib_runtime_initialized,False


**说明**：qlib 运行时会在第 6 节显式调用 `init_qlib_runtime()` 初始化。

## 3. 参数区

默认走 `watchlist`，但可以直接切换到 `csi300`、`csi500` 或 `merged_csi300_500`。`watchlist` 默认读取 `data/watchlist.json`；如果你只想临时跑一组股票，可以设置 `WATCHLIST_SYMBOLS_OVERRIDE`。

In [3]:
DATE_TAG = pd.Timestamp.now(tz="Asia/Shanghai").strftime("%Y%m%d")

UNIVERSE_PROFILE = "csi300"
UNIVERSE_OPTIONS = ("watchlist", "csi300", "csi500", "merged_csi300_500")
ACTIVE_FEATURE_GROUPS = (
    "technical_core",
    "valuation_absolute",
    "valuation_percentile",
    "industry_valuation_context",
    "quality_summary",
    "fscore_components",
    "ttm_profitability",
)
RUN_EXPORT = "auto_if_missing"
RUN_PUBLISH = False
INDUSTRY_NORMALIZATION = "l1_weekly_robust"
WATCHLIST_SYMBOLS_OVERRIDE = None

START_DATE = None
END_DATE = None
BATCH_SIZE = 300
EVAL_COUNT = 52
TRAIN_WEEKS = 260
VALID_WEEKS = 52
STEP_WEEKS = 1
TOPK = 10

BACKTEST_MODE = "weekly_net"
EXECUTION_PRICE = "next_open"
EXECUTION_LAG_STEPS = 1
DEFAULT_WEEKLY_NET_CONFIG = default_weekly_net_backtest_config()
BROKER_COMMISSION_RATE = DEFAULT_WEEKLY_NET_CONFIG.broker_commission_rate
EXCHANGE_FEE_RATE = DEFAULT_WEEKLY_NET_CONFIG.exchange_fee_rate
TRANSFER_FEE_RATE = DEFAULT_WEEKLY_NET_CONFIG.transfer_fee_rate
STAMP_DUTY_SELL_RATE = DEFAULT_WEEKLY_NET_CONFIG.stamp_duty_sell_rate
IMPACT_COST_RATE = DEFAULT_WEEKLY_NET_CONFIG.impact_cost_rate
MIN_COMMISSION = DEFAULT_WEEKLY_NET_CONFIG.min_commission
TRADE_UNIT = 100
ENABLE_STOP_LOSS = False
STOP_LOSS_THRESHOLD = DEFAULT_WEEKLY_NET_CONFIG.stop_loss_threshold
STOP_LOSS_MODE = "weekly_low" if ENABLE_STOP_LOSS else "off"

LONG_WALK_FORWARD_ENABLED = True
LONG_WALK_FORWARD_START_DATE = "2016-01-01"
LONG_WALK_FORWARD_END_DATE = "2026-03-27"
LONG_WALK_FORWARD_TRAIN_WEEKS = 260
LONG_WALK_FORWARD_VALID_WEEKS = 52
LONG_WALK_FORWARD_STEP_WEEKS = 1
LONG_WALK_FORWARD_EVAL_COUNT = 0

MODEL_PARAMS = {
    "loss": "mse",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "lambda_l2": 1,
    "num_boost_round": 200,
    "early_stopping_rounds": 50,
}

backtest_config = BacktestTradingConfig(
    broker_commission_rate=BROKER_COMMISSION_RATE,
    exchange_fee_rate=EXCHANGE_FEE_RATE,
    transfer_fee_rate=TRANSFER_FEE_RATE,
    stamp_duty_sell_rate=STAMP_DUTY_SELL_RATE,
    impact_cost_rate=IMPACT_COST_RATE,
    min_commission=MIN_COMMISSION,
    trade_unit=TRADE_UNIT,
    stop_loss_mode=STOP_LOSS_MODE,
    stop_loss_threshold=STOP_LOSS_THRESHOLD,
)
qlib_exchange_preview = pd.DataFrame([backtest_config.qlib_exchange_kwargs(deal_price="$open")])

EXPERIMENT_NAME = "valueinvesting_weekly_qlib_workflow_by_code"
PANEL_PATH = _panel_path_for_universe(UNIVERSE_PROFILE)
OUTPUT_DIR = _output_dir_for_universe(UNIVERSE_PROFILE, DATE_TAG)
CANDIDATE_MODEL_ID = f"{('csi300500' if UNIVERSE_PROFILE == 'merged_csi300_500' else UNIVERSE_PROFILE)}-weekly-lgbm-explicit-{DATE_TAG}"

assert UNIVERSE_PROFILE in UNIVERSE_OPTIONS, f"Unsupported universe: {UNIVERSE_PROFILE}"
assert BACKTEST_MODE == "weekly_net", "当前 notebook 已按 A 股净收益基线实现，仅支持 weekly_net。"
assert EXECUTION_PRICE == "next_open", "显式工作流默认采用次期开盘成交，避免同周前视。"

parameter_frame = pd.DataFrame(
    [
        ("UNIVERSE_PROFILE", UNIVERSE_PROFILE),
        ("UNIVERSE_OPTIONS", ", ".join(UNIVERSE_OPTIONS)),
        ("ACTIVE_FEATURE_GROUPS", ", ".join(ACTIVE_FEATURE_GROUPS)),
        ("RUN_EXPORT", RUN_EXPORT),
        ("RUN_PUBLISH", RUN_PUBLISH),
        ("INDUSTRY_NORMALIZATION", INDUSTRY_NORMALIZATION),
        ("WATCHLIST_SYMBOLS_OVERRIDE", WATCHLIST_SYMBOLS_OVERRIDE),
        ("PANEL_PATH", PANEL_PATH),
        ("OUTPUT_DIR", OUTPUT_DIR),
        ("EVAL_COUNT", EVAL_COUNT),
        ("TRAIN_WEEKS", TRAIN_WEEKS),
        ("VALID_WEEKS", VALID_WEEKS),
        ("STEP_WEEKS", STEP_WEEKS),
        ("TOPK", TOPK),
        ("BACKTEST_MODE", BACKTEST_MODE),
        ("EXECUTION_PRICE", EXECUTION_PRICE),
        ("EXECUTION_LAG_STEPS", EXECUTION_LAG_STEPS),
        ("BROKER_COMMISSION_RATE", BROKER_COMMISSION_RATE),
        ("EXCHANGE_FEE_RATE", EXCHANGE_FEE_RATE),
        ("TRANSFER_FEE_RATE", TRANSFER_FEE_RATE),
        ("STAMP_DUTY_SELL_RATE", STAMP_DUTY_SELL_RATE),
        ("IMPACT_COST_RATE", IMPACT_COST_RATE),
        ("MIN_COMMISSION", MIN_COMMISSION),
        ("TRADE_UNIT", TRADE_UNIT),
        ("ENABLE_STOP_LOSS", ENABLE_STOP_LOSS),
        ("STOP_LOSS_MODE", STOP_LOSS_MODE),
        ("STOP_LOSS_THRESHOLD", STOP_LOSS_THRESHOLD),
        ("LONG_WALK_FORWARD_ENABLED", LONG_WALK_FORWARD_ENABLED),
        ("LONG_WALK_FORWARD_START_DATE", LONG_WALK_FORWARD_START_DATE),
        ("LONG_WALK_FORWARD_END_DATE", LONG_WALK_FORWARD_END_DATE),
        ("LONG_WALK_FORWARD_TRAIN_WEEKS", LONG_WALK_FORWARD_TRAIN_WEEKS),
        ("LONG_WALK_FORWARD_VALID_WEEKS", LONG_WALK_FORWARD_VALID_WEEKS),
        ("LONG_WALK_FORWARD_STEP_WEEKS", LONG_WALK_FORWARD_STEP_WEEKS),
        ("LONG_WALK_FORWARD_EVAL_COUNT", LONG_WALK_FORWARD_EVAL_COUNT),
        ("MODEL_PARAMS", MODEL_PARAMS),
        ("CANDIDATE_MODEL_ID", CANDIDATE_MODEL_ID),
    ],
    columns=["parameter", "value"],
)
display(parameter_frame)
display(Markdown("**qlib Exchange 成本参数预览**"))
display(qlib_exchange_preview)


,parameter,value
0,UNIVERSE_PROFILE,csi300
1,UNIVERSE_OPTIONS,"watchlist, csi300, csi500, merged_csi300_500"
2,ACTIVE_FEATURE_GROUPS,"technical_core, valuation_absolute, valuation_..."
3,RUN_EXPORT,auto_if_missing
4,RUN_PUBLISH,False
5,INDUSTRY_NORMALIZATION,l1_weekly_robust
6,WATCHLIST_SYMBOLS_OVERRIDE,None
7,PANEL_PATH,artifacts/panels/csi300_weekly.csv
8,OUTPUT_DIR,artifacts/evaluations/csi300_workflo...
9,EVAL_COUNT,52


**qlib Exchange 成本参数预览**

,deal_price,open_cost,close_cost,impact_cost,min_cost,trade_unit
0,$open,0.000244,0.000744,0.000500,5.000000,100


## 4. Universe 解析与 Panel 导出

这一节负责把 universe 配置转换成实际股票池，并导出/加载周频特征 panel。`watchlist` 默认读取 `data/watchlist.json`；如果它为空，直接提示改用 `WATCHLIST_SYMBOLS_OVERRIDE` 或切换 universe。

In [4]:
watchlist_path = PROJECT_ROOT / "data" / "watchlist.json"
watchlist_records = json.loads(watchlist_path.read_text(encoding="utf-8")) if watchlist_path.exists() else []
watchlist_symbols = sorted(
    {
        str(record.get("ticker") or "").strip().upper()
        for record in watchlist_records
        if str(record.get("ticker") or "").strip()
    }
)
override_symbols = (
    [str(symbol).strip().upper() for symbol in WATCHLIST_SYMBOLS_OVERRIDE if str(symbol).strip()]
    if WATCHLIST_SYMBOLS_OVERRIDE
    else None
)

if UNIVERSE_PROFILE == "watchlist":
    display(
        pd.DataFrame(
            [
                {
                    "watchlist_path": str(watchlist_path),
                    "watchlist_symbol_count": len(watchlist_symbols),
                    "override_symbol_count": 0 if override_symbols is None else len(override_symbols),
                }
            ]
        )
    )
    assert watchlist_symbols or override_symbols, (
        "watchlist 为空，请更新 data/watchlist.json、设置 WATCHLIST_SYMBOLS_OVERRIDE，或切换到其他 universe。"
    )

panel_result = export_or_load_panel(
    panel_path=PANEL_PATH,
    universe_profile=UNIVERSE_PROFILE,
    run_export=RUN_EXPORT,
    start_date=START_DATE,
    end_date=END_DATE,
    symbols=override_symbols if UNIVERSE_PROFILE == "watchlist" else None,
    batch_size=BATCH_SIZE,
    return_panel=True,
)
panel = panel_result["panel"].copy()

assert panel is not None and not panel.empty, "panel 为空，请先检查 universe 配置、FDH 数据和导出参数。"

display(pd.DataFrame([{"action": panel_result["action"], "panel_path": str(panel_result["path"]), **panel_result["summary"]}]))

preview_columns = [
    column
    for column in [
        "datetime",
        "instrument",
        "in_csi300",
        "in_csi500",
        "l1_name",
        "close",
        "pe_ttm",
        "f_score",
        LABEL_COLUMN,
    ]
    if column in panel.columns
]
display(panel[preview_columns].head(10))

,action,panel_path,rows,instrument_count,start_date,end_date,csi300_rows,csi500_rows,csi300_instruments,csi500_instruments
0,loaded_existing,/Volumes/Repository/Projects/TradingNexus/Valu...,150352,607,2016-01-29,2026-03-27,150352,0,607,0


,datetime,instrument,in_csi300,l1_name,close,pe_ttm,f_score,label_excess_return_4w
1234,2016-01-29,000001.SZ,True,银行,6.389834,6.549200,6.000000,-0.009528
1235,2016-02-05,000001.SZ,True,银行,6.338715,6.496800,6.000000,0.024563
1236,2016-02-19,000001.SZ,True,银行,6.415393,6.575400,6.000000,0.021900
1237,2016-02-26,000001.SZ,True,银行,6.255647,6.411700,6.000000,-0.013770
1238,2016-03-04,000001.SZ,True,银行,6.645427,6.811200,6.000000,-0.075415
1239,2016-03-11,000001.SZ,True,银行,6.492071,6.648800,6.000000,-0.077125
1240,2016-03-18,000001.SZ,True,银行,6.734885,6.897500,6.000000,-0.029510
1241,2016-03-25,000001.SZ,True,银行,6.766834,6.930200,6.000000,-0.000585
1242,2016-04-01,000001.SZ,True,银行,6.811563,6.976000,6.000000,0.009243
1243,2016-04-08,000001.SZ,True,银行,6.754054,6.917100,6.000000,0.017332


## 5. 特征组选择、行业标准化与 Panel EDA

这里把特征组、预过滤、行业内标准化和基础 EDA 放在一起，方便明确“这次训练究竟吃了哪些列”。

In [5]:
active_feature_columns = compose_feature_columns(ACTIVE_FEATURE_GROUPS)
feature_group_frame = pd.DataFrame(
    [
        {
            "feature_group": group_name,
            "feature_count": len(FEATURE_GROUP_COLUMNS[group_name]),
            "features": ", ".join(FEATURE_GROUP_COLUMNS[group_name]),
        }
        for group_name in ACTIVE_FEATURE_GROUPS
    ]
)
display(feature_group_frame)

filtered_feature_columns, feature_stats, corr_marks = prefilter_feature_columns(panel, active_feature_columns)
normalized_panel = apply_industry_normalization(
    panel,
    feature_columns=filtered_feature_columns,
    method=INDUSTRY_NORMALIZATION,
)

assert filtered_feature_columns, "预过滤后没有可用特征，请先检查 panel 质量和特征组配置。"

display(
    pd.DataFrame(
        [
            {
                "selected_feature_count": len(active_feature_columns),
                "filtered_feature_count": len(filtered_feature_columns),
                "removed_feature_count": len(active_feature_columns) - len(filtered_feature_columns),
                "industry_normalization": INDUSTRY_NORMALIZATION,
            }
        ]
    )
)

coverage_aggs = {
    "rows": ("instrument", "size"),
    "instruments": ("instrument", "nunique"),
}
if "in_csi300" in panel.columns:
    coverage_aggs["csi300_rows"] = ("in_csi300", "sum")
if "in_csi500" in panel.columns:
    coverage_aggs["csi500_rows"] = ("in_csi500", "sum")

coverage_by_date = panel.groupby("datetime").agg(**coverage_aggs).reset_index()
coverage_fig = px.line(
    coverage_by_date,
    x="datetime",
    y=[column for column in coverage_by_date.columns if column != "datetime"],
    title="panel 覆盖概览",
)
coverage_fig.update_layout(height=500)
coverage_fig.show()

display(feature_stats.head(20))
missing_fig = px.bar(
    feature_stats.sort_values("missing_ratio", ascending=False),
    x="feature",
    y="missing_ratio",
    color="keep",
    title="特征缺失率与保留结果",
)
missing_fig.update_layout(height=500, xaxis_tickangle=-45)
missing_fig.show()

if not corr_marks.empty:
    display(corr_marks.head(20))
    heatmap_features = list(
        dict.fromkeys(corr_marks["left_feature"].head(6).tolist() + corr_marks["right_feature"].head(6).tolist())
    )
    corr_source = normalized_panel[heatmap_features].apply(pd.to_numeric, errors="coerce")
    corr_source = corr_source.fillna(corr_source.median(numeric_only=True))
    corr_matrix = corr_source.corr()
    corr_fig = px.imshow(corr_matrix, text_auto=".2f", aspect="auto", title="高相关候选热图（截取）")
    corr_fig.update_layout(height=500)
    corr_fig.show()
else:
    display(Markdown("**高相关候选**：当前预过滤阶段没有发现超过阈值的高相关特征。"))

,feature_group,feature_count,features
0,technical_core,11,"signal_strength, signal_grade_num, ma20, ma50,..."
1,valuation_absolute,5,"pe_ttm, pb, ps_ttm, dv_ttm, peg"
2,valuation_percentile,3,"pe_ttm_pct_1250d, pb_pct_1250d, ps_ttm_pct_1250d"
3,industry_valuation_context,2,"core_indicator_pct_1250d, core_indicator_indus..."
4,quality_summary,5,"f_score, roe_5y_avg, ni_cfo_corr_3y, debt_rati..."
5,fscore_components,9,"f_roa, f_cfo, f_delta_roa, f_accrual, f_delta_..."
6,ttm_profitability,5,"roa_ttm, cfo_ttm, ni_ttm, gpm_ttm, at_ttm"


,selected_feature_count,filtered_feature_count,removed_feature_count,industry_normalization
0,40,36,4,l1_weekly_robust


,feature,missing_ratio,overall_std,median_cross_section_std,keep
0,amount,0.000000,"6,198,859.474272","3,919,583.415470",True
1,volume,0.000000,"4,058,068.547208","3,005,640.638429",True
2,atr,0.032969,6.435629,5.482524,True
3,rsi,0.032969,11.644335,9.864848,True
4,ma20,0.042613,86.955484,93.040616,True
5,dea,0.071333,5.003786,3.526645,True
6,dif,0.071333,5.308200,3.668892,True
7,macd_hist,0.071333,3.187736,2.077736,True
8,f_accrual,0.080538,0.467189,0.466477,True
9,f_cfo,0.080538,0.310023,0.285591,True


,left_feature,right_feature,abs_corr
0,ma20,ma50,0.981705
1,dif,dea,0.953926
2,ma20,atr,0.912947


## 6. 显式训练节点

这一节直接显式实例化 qlib 的 `StaticDataLoader`、`DataHandlerLP`、`DatasetH` 和 `LGBModel`，保留官方 notebook 的工作流感受。

In [6]:
init_qlib_runtime(exp_name=EXPERIMENT_NAME)

with suppress_external_output():
    from qlib.contrib.model.gbdt import LGBModel
    from qlib.data.dataset import DatasetH
    from qlib.data.dataset.handler import DataHandlerLP
    from qlib.data.dataset.loader import StaticDataLoader
    from qlib.workflow import R

latest_feature_date = pd.to_datetime(normalized_panel["datetime"]).max()
training_panel = normalized_panel.loc[pd.to_datetime(normalized_panel["datetime"]) <= latest_feature_date].copy()
segments = build_segments(training_panel, feature_date=latest_feature_date)
qlib_frame, used_feature_columns = build_training_frame(
    training_panel,
    feature_columns=filtered_feature_columns,
    label_column=LABEL_COLUMN,
)

loader = StaticDataLoader(qlib_frame)
handler = DataHandlerLP(data_loader=loader, infer_processors=[], learn_processors=[])
dataset = DatasetH(handler=handler, segments=segments)
model = LGBModel(**MODEL_PARAMS)

with R.start(experiment_name=EXPERIMENT_NAME):
    with suppress_external_output():
        model.fit(dataset, verbose_eval=False)
    explicit_train_recorder_id = R.get_recorder().id

display(
    pd.DataFrame(
        [
            {
                "latest_feature_date": str(pd.Timestamp(latest_feature_date).date()),
                "used_feature_count": len(used_feature_columns),
                "recorder_id": explicit_train_recorder_id,
            }
        ]
    )
)
display(
    pd.DataFrame(
        [{"segment": segment_name, "start": window[0], "end": window[1]} for segment_name, window in segments.items()]
    )
)

feature_importance_frame = pd.DataFrame(
    {"feature": used_feature_columns, "importance": model.model.feature_importance()}
).sort_values("importance", ascending=False).reset_index(drop=True)
display(feature_importance_frame.head(20))

/Volumes/Repository/Projects/TradingNexus/QlibResearch/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning:

The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.



,latest_feature_date,used_feature_count,recorder_id
0,2026-03-27,36,4f6612a86bc846dfaa5b831175153ec5


,segment,start,end
0,train,2016-01-29,2024-03-01
1,valid,2024-03-08,2026-02-27
2,test,2026-03-27,2026-03-27


,feature,importance
0,macd_hist,10
1,rsi,10
2,pb,9
3,ni_ttm,8
4,atr,8
5,ps_ttm,8
6,dea,7
7,at_ttm,6
8,roa_ttm,5
9,current_ratio,5


## 7. 显式预测节点

这里直接对测试段做 `model.predict()`，再把结果整理成 notebook 内分析表和线上可发布的快照表。

In [7]:
with suppress_external_output():
    raw_prediction = model.predict(dataset, segment="test")

prediction_frame = raw_prediction.reset_index()
prediction_value_column = prediction_frame.columns[-1]
prediction_frame = prediction_frame.rename(columns={prediction_value_column: "score"})
prediction_frame["datetime"] = pd.to_datetime(prediction_frame["datetime"])
prediction_frame["instrument"] = prediction_frame["instrument"].astype(str)
prediction_frame["used_feature_count"] = len(used_feature_columns)

latest_prediction_with_labels = attach_future_labels_to_prediction(
    prediction_frame[["datetime", "instrument", "score", "used_feature_count"]].copy(),
    normalized_panel,
    latest_feature_date,
).sort_values("score", ascending=False).reset_index(drop=True)

score_snapshot_frame = latest_prediction_with_labels[["instrument", "score"]].copy()
score_snapshot_frame = score_snapshot_frame.rename(columns={"instrument": "code", "score": "qlib_score"})
score_snapshot_frame = score_snapshot_frame.sort_values("qlib_score", ascending=False).reset_index(drop=True)
score_snapshot_frame["qlib_rank"] = score_snapshot_frame.index + 1
score_snapshot_frame["pred_return_4w"] = score_snapshot_frame["qlib_score"]

display(
    pd.DataFrame(
        [
            {
                "latest_feature_date": str(pd.Timestamp(latest_feature_date).date()),
                "prediction_count": len(score_snapshot_frame),
                "top_score": None if score_snapshot_frame.empty else float(score_snapshot_frame["qlib_score"].iloc[0]),
            }
        ]
    )
)

display(Markdown(f"**最新预测日期**：`{pd.Timestamp(latest_feature_date).date()}`"))
display(
    latest_prediction_with_labels[
        [
            column
            for column in [
                "instrument",
                "score",
                "future_return_4w",
                LABEL_COLUMN,
                "l1_name",
                "in_csi300",
                "in_csi500",
            ]
            if column in latest_prediction_with_labels.columns
        ]
    ].head(20)
)
display(score_snapshot_frame.head(10))

,latest_feature_date,prediction_count,top_score
0,2026-03-27,300,0.001827


**最新预测日期**：`2026-03-27`

,instrument,score,future_return_4w,label_excess_return_4w,l1_name,in_csi300
0,688506.SH,0.001827,NaN,NaN,医药生物,True
1,601100.SH,0.001318,NaN,NaN,机械设备,True
2,300033.SZ,0.001200,NaN,NaN,计算机,True
3,601888.SH,-0.000769,NaN,NaN,商贸零售,True
4,603893.SH,-0.000769,NaN,NaN,电子,True
5,688012.SH,-0.000769,NaN,NaN,电子,True
6,601225.SH,-0.000852,NaN,NaN,煤炭,True
7,600233.SH,-0.001468,NaN,NaN,交通运输,True
8,600660.SH,-0.003067,NaN,NaN,汽车,True
9,300760.SZ,-0.003773,NaN,NaN,医药生物,True


,code,qlib_score,qlib_rank,pred_return_4w
0,688506.SH,0.001827,1,0.001827
1,601100.SH,0.001318,2,0.001318
2,300033.SZ,0.001200,3,0.001200
3,688012.SH,-0.000769,4,-0.000769
4,601888.SH,-0.000769,5,-0.000769
5,603893.SH,-0.000769,6,-0.000769
6,601225.SH,-0.000852,7,-0.000852
7,600233.SH,-0.001468,8,-0.001468
8,600660.SH,-0.003067,9,-0.003067
9,300760.SZ,-0.003773,10,-0.003773


## 8. 显式滚动回测节点

这一节不用 `run_convergence_workflow()`，而是直接调用周频评估辅助函数，先跑最近 1 年的快速滚动评估，再按参数决定是否追加长期 walk-forward 稳健性测试。

In [8]:
def run_explicit_rolling_backtest(
    evaluation_name,
    panel,
    feature_columns,
    train_weeks,
    valid_weeks,
    eval_count,
    step_weeks,
    start_date,
    end_date,
    topk,
    model_params,
    backtest_mode,
    execution_price_label,
    execution_lag_steps,
    trading_config,
):
    eval_dates = select_evaluation_dates(
        panel=panel,
        train_weeks=train_weeks,
        valid_weeks=valid_weeks,
        eval_count=eval_count,
        step_weeks=step_weeks,
        start_date=start_date,
        end_date=end_date,
    )
    if not eval_dates:
        raise ValueError(f"{evaluation_name} 没有可用的滚动评估日期，请检查标签覆盖和窗口参数。")

    shared_model_params = {
        key: value
        for key, value in model_params.items()
        if key not in {"num_boost_round", "early_stopping_rounds"}
    }
    prediction_frames = []
    detail_rows = []
    for feature_date in eval_dates:
        one_score = fit_predict_one_date(
            panel=panel,
            feature_date=feature_date,
            feature_columns=feature_columns,
            train_weeks=train_weeks,
            valid_weeks=valid_weeks,
            num_boost_round=int(model_params["num_boost_round"]),
            early_stopping_rounds=int(model_params["early_stopping_rounds"]),
            model_params=shared_model_params,
        )
        merged = attach_future_labels_to_prediction(one_score.copy(), panel, feature_date)
        merged["recipe"] = evaluation_name
        merged["evaluation_name"] = evaluation_name
        prediction_frames.append(merged)

        metrics = evaluate_prediction_frame(merged, topk=topk)
        metrics.update(
            {
                "recipe": evaluation_name,
                "evaluation_name": evaluation_name,
                "feature_date": pd.Timestamp(feature_date),
                "used_feature_count": int(merged["used_feature_count"].max()),
            }
        )
        detail_rows.append(metrics)

    predictions = pd.concat(prediction_frames, ignore_index=True)
    details = pd.DataFrame(detail_rows).sort_values("feature_date").reset_index(drop=True)
    summary = summarize_details(details, group_columns=("recipe",))
    if not summary.empty:
        summary["evaluation_name"] = evaluation_name
        summary["train_weeks"] = train_weeks
        summary["valid_weeks"] = valid_weeks
        summary["step_weeks"] = step_weeks
        summary["eval_count"] = eval_count

    quote_frames = build_backtest_price_frames(panel, eval_dates=eval_dates)
    signal_matrix = build_signal_matrix(
        predictions,
        quote_frames.execution_price,
        topk=topk,
        execution_lag_steps=execution_lag_steps,
    )
    equity_curve, backtest_metrics = run_strategy_backtest(
        quote_frames.mark_price,
        signal_matrix,
        execution_price_frame=quote_frames.execution_price,
        stop_price_frame=quote_frames.stop_price if trading_config.stop_loss_mode != "off" else None,
        trading_config=trading_config,
    )
    backtest_metrics["backtest_mode"] = backtest_mode
    backtest_metrics["execution_price"] = execution_price_label
    equity_curve_frame = equity_curve.reset_index().rename(columns={"date": "datetime"}) if not equity_curve.empty else pd.DataFrame()
    if not equity_curve_frame.empty:
        equity_curve_frame["evaluation_name"] = evaluation_name
        equity_curve_frame["series"] = "all"

    available_slice_flags = {
        slice_name: flag_column
        for slice_name, flag_column in SLICE_FLAG_COLUMNS.items()
        if flag_column in predictions.columns and predictions[flag_column].fillna(False).any()
    }
    slice_summaries = []
    slice_equity_frames = []
    for slice_name, flag_column in available_slice_flags.items():
        slice_predictions = predictions.loc[predictions[flag_column].fillna(False)].copy()
        if slice_predictions.empty:
            continue

        slice_detail_rows = []
        for one_feature_date, frame in slice_predictions.groupby("feature_date"):
            slice_metrics = evaluate_prediction_frame(frame, topk=topk)
            slice_metrics.update(
                {
                    "recipe": evaluation_name,
                    "evaluation_name": evaluation_name,
                    "slice": slice_name,
                    "feature_date": pd.Timestamp(one_feature_date),
                    "used_feature_count": int(frame["used_feature_count"].max()),
                }
            )
            slice_detail_rows.append(slice_metrics)

        slice_detail_frame = pd.DataFrame(slice_detail_rows).sort_values("feature_date").reset_index(drop=True)
        slice_summary = summarize_details(slice_detail_frame, group_columns=("recipe", "slice"))
        slice_symbols = sorted(slice_predictions["instrument"].astype(str).unique().tolist())
        slice_quote_frames = build_backtest_price_frames(panel, eval_dates=eval_dates, symbols=slice_symbols)
        slice_signal_matrix = build_signal_matrix(
            slice_predictions,
            slice_quote_frames.execution_price,
            topk=topk,
            execution_lag_steps=execution_lag_steps,
        )
        slice_equity_curve, slice_backtest_metrics = run_strategy_backtest(
            slice_quote_frames.mark_price,
            slice_signal_matrix,
            execution_price_frame=slice_quote_frames.execution_price,
            stop_price_frame=slice_quote_frames.stop_price if trading_config.stop_loss_mode != "off" else None,
            trading_config=trading_config,
        )
        slice_backtest_metrics["backtest_mode"] = backtest_mode
        slice_backtest_metrics["execution_price"] = execution_price_label

        if not slice_summary.empty:
            for metric_name, metric_value in slice_backtest_metrics.items():
                slice_summary[metric_name] = metric_value
            slice_summary["evaluation_name"] = evaluation_name
            slice_summary["train_weeks"] = train_weeks
            slice_summary["valid_weeks"] = valid_weeks
            slice_summary["step_weeks"] = step_weeks
            slice_summary["eval_count"] = eval_count
            slice_summaries.append(slice_summary)

        if not slice_equity_curve.empty:
            slice_equity_frame = slice_equity_curve.reset_index().rename(columns={"date": "datetime"})
            slice_equity_frame["slice"] = slice_name
            slice_equity_frame["evaluation_name"] = evaluation_name
            slice_equity_frame["series"] = slice_name
            slice_equity_frames.append(slice_equity_frame)

    slice_summary_frame = pd.concat(slice_summaries, ignore_index=True) if slice_summaries else pd.DataFrame()
    slice_equity_curve_frame = pd.concat(slice_equity_frames, ignore_index=True) if slice_equity_frames else pd.DataFrame()
    config_frame = pd.DataFrame(
        [
            {
                "evaluation_name": evaluation_name,
                "rolling_eval_dates": len(eval_dates),
                "first_eval_date": str(pd.Timestamp(eval_dates[0]).date()),
                "last_eval_date": str(pd.Timestamp(eval_dates[-1]).date()),
                "train_weeks": train_weeks,
                "valid_weeks": valid_weeks,
                "step_weeks": step_weeks,
                "eval_count": eval_count,
                "start_date": start_date,
                "end_date": end_date,
                "topk": topk,
                "backtest_mode": backtest_mode,
                "execution_price": execution_price_label,
                "execution_lag_steps": execution_lag_steps,
                "trade_unit": trading_config.trade_unit,
                "stop_loss_mode": trading_config.stop_loss_mode,
            }
        ]
    )
    return {
        "evaluation_name": evaluation_name,
        "eval_dates": eval_dates,
        "predictions": predictions,
        "details": details,
        "summary": summary,
        "execution_price_frame": quote_frames.execution_price,
        "mark_price_frame": quote_frames.mark_price,
        "signal_matrix": signal_matrix,
        "equity_curve": equity_curve,
        "equity_curve_frame": equity_curve_frame,
        "backtest_metrics": backtest_metrics,
        "slice_summary": slice_summary_frame,
        "slice_equity_curve_frame": slice_equity_curve_frame,
        "config_frame": config_frame,
    }


rolling_result = run_explicit_rolling_backtest(
    evaluation_name="recent_1y_weekly_walk_forward",
    panel=normalized_panel,
    feature_columns=filtered_feature_columns,
    train_weeks=TRAIN_WEEKS,
    valid_weeks=VALID_WEEKS,
    eval_count=EVAL_COUNT,
    step_weeks=STEP_WEEKS,
    start_date=START_DATE,
    end_date=END_DATE,
    topk=TOPK,
    model_params=MODEL_PARAMS,
    backtest_mode=BACKTEST_MODE,
    execution_price_label=EXECUTION_PRICE,
    execution_lag_steps=EXECUTION_LAG_STEPS,
    trading_config=backtest_config,
)

rolling_eval_dates = rolling_result["eval_dates"]
rolling_predictions = rolling_result["predictions"]
rolling_details = rolling_result["details"]
rolling_summary = rolling_result["summary"]
execution_price_frame = rolling_result["execution_price_frame"]
mark_price_frame = rolling_result["mark_price_frame"]
signal_matrix = rolling_result["signal_matrix"]
equity_curve = rolling_result["equity_curve"]
equity_curve_frame = rolling_result["equity_curve_frame"]
backtest_metrics = rolling_result["backtest_metrics"]
rolling_backtest_metric_frame = _build_backtest_metric_frame("recent_1y_weekly_walk_forward", backtest_metrics)
slice_summary_frame = rolling_result["slice_summary"]
slice_equity_curve_frame = rolling_result["slice_equity_curve_frame"]

display(Markdown("**快速滚动评估摘要**"))
display(rolling_result["config_frame"])
display(rolling_summary)
display(rolling_backtest_metric_frame)
if slice_summary_frame.empty:
    display(Markdown("**切片分析**：当前快速滚动评估无 `in_csi300` / `in_csi500` 切片结果，已跳过切片回测。"))
else:
    display(slice_summary_frame)

if LONG_WALK_FORWARD_ENABLED:
    walk_forward_result = run_explicit_rolling_backtest(
        evaluation_name="long_walk_forward",
        panel=normalized_panel,
        feature_columns=filtered_feature_columns,
        train_weeks=LONG_WALK_FORWARD_TRAIN_WEEKS,
        valid_weeks=LONG_WALK_FORWARD_VALID_WEEKS,
        eval_count=LONG_WALK_FORWARD_EVAL_COUNT,
        step_weeks=LONG_WALK_FORWARD_STEP_WEEKS,
        start_date=LONG_WALK_FORWARD_START_DATE,
        end_date=LONG_WALK_FORWARD_END_DATE,
        topk=TOPK,
        model_params=MODEL_PARAMS,
        backtest_mode=BACKTEST_MODE,
        execution_price_label=EXECUTION_PRICE,
        execution_lag_steps=EXECUTION_LAG_STEPS,
        trading_config=backtest_config,
    )
    walk_forward_eval_dates = walk_forward_result["eval_dates"]
    walk_forward_predictions = walk_forward_result["predictions"]
    walk_forward_details = walk_forward_result["details"]
    walk_forward_summary = walk_forward_result["summary"]
    walk_forward_execution_price_frame = walk_forward_result["execution_price_frame"]
    walk_forward_mark_price_frame = walk_forward_result["mark_price_frame"]
    walk_forward_signal_matrix = walk_forward_result["signal_matrix"]
    walk_forward_equity_curve = walk_forward_result["equity_curve"]
    walk_forward_equity_curve_frame = walk_forward_result["equity_curve_frame"]
    walk_forward_backtest_metrics = walk_forward_result["backtest_metrics"]
    walk_forward_backtest_metric_frame = _build_backtest_metric_frame("long_walk_forward", walk_forward_backtest_metrics)
    walk_forward_slice_summary_frame = walk_forward_result["slice_summary"]
    walk_forward_slice_equity_curve_frame = walk_forward_result["slice_equity_curve_frame"]

    display(Markdown("**长期 Walk-forward 摘要**"))
    display(walk_forward_result["config_frame"])
    display(walk_forward_summary)
    display(walk_forward_backtest_metric_frame)
    if walk_forward_slice_summary_frame.empty:
        display(Markdown("**长期 Walk-forward 切片分析**：当前无 `in_csi300` / `in_csi500` 切片结果。"))
    else:
        display(walk_forward_slice_summary_frame)
else:
    walk_forward_result = None
    walk_forward_eval_dates = []
    walk_forward_predictions = pd.DataFrame()
    walk_forward_details = pd.DataFrame()
    walk_forward_summary = pd.DataFrame()
    walk_forward_execution_price_frame = pd.DataFrame()
    walk_forward_mark_price_frame = pd.DataFrame()
    walk_forward_signal_matrix = pd.DataFrame()
    walk_forward_equity_curve = pd.DataFrame()
    walk_forward_equity_curve_frame = pd.DataFrame()
    walk_forward_backtest_metrics = {}
    walk_forward_backtest_metric_frame = pd.DataFrame()
    walk_forward_slice_summary_frame = pd.DataFrame()
    walk_forward_slice_equity_curve_frame = pd.DataFrame()
    display(Markdown("**长期 Walk-forward**：已禁用。把 `LONG_WALK_FORWARD_ENABLED=True`，并设置 `LONG_WALK_FORWARD_START_DATE` / `LONG_WALK_FORWARD_END_DATE` 后可执行跨多年稳健性测试。"))


**快速滚动评估摘要**

,evaluation_name,rolling_eval_dates,first_eval_date,last_eval_date,train_weeks,valid_weeks,step_weeks,eval_count,start_date,end_date,topk,backtest_mode,execution_price,execution_lag_steps,trade_unit,stop_loss_mode
0,recent_1y_weekly_walk_forward,52,2025-02-28,2026-02-27,260,52,1,52,None,None,10,weekly_net,next_open,1,100,off


,recipe,evaluation_dates,coverage_mean,rank_ic_mean,rank_ic_std,rank_ic_ir,ic_mean,ic_std,topk_mean_return_4w,topk_mean_excess_return_4w,topk_hit_rate,universe_mean_return_4w,used_feature_count,evaluation_name,train_weeks,valid_weeks,step_weeks,eval_count
0,recent_1y_weekly_walk_forward,52,298.903846,0.031613,0.153285,0.206237,-0.003055,0.129674,0.024613,0.008313,0.561538,0.015245,36,recent_1y_weekly_walk_forward,260,52,1,52


,evaluation_name,strategy_total_return,strategy_cagr,strategy_max_drawdown,strategy_weekly_sharpe,strategy_total_cost,strategy_commission,strategy_exchange_fee,strategy_transfer_fee,strategy_stamp_duty,strategy_impact_cost,strategy_gross_trade_value,strategy_turnover_ratio_total,strategy_trade_count,backtest_mode,execution_price
0,recent_1y_weekly_walk_forward,0.283559,0.284660,-0.085461,1.441517,"63,867.758213","13,215.114864","2,185.159967",640.809375,"15,786.205277","32,040.468731","64,080,937.461500",60.045271,734.000000,weekly_net,next_open


,recipe,slice,evaluation_dates,coverage_mean,rank_ic_mean,rank_ic_std,rank_ic_ir,ic_mean,ic_std,topk_mean_return_4w,topk_mean_excess_return_4w,topk_hit_rate,universe_mean_return_4w,used_feature_count,strategy_total_return,strategy_cagr,strategy_max_drawdown,strategy_weekly_sharpe,strategy_gross_trade_value,strategy_total_cost,strategy_commission,strategy_exchange_fee,strategy_transfer_fee,strategy_stamp_duty,strategy_impact_cost,strategy_turnover_ratio_mean,strategy_turnover_ratio_total,strategy_trade_count,backtest_mode,execution_price,evaluation_name,train_weeks,valid_weeks,step_weeks,eval_count
0,recent_1y_weekly_walk_forward,csi300,52,298.903846,0.031613,0.153285,0.206237,-0.003055,0.129674,0.024613,0.008313,0.561538,0.015245,36,0.283559,0.284660,-0.085461,1.441517,"64,080,937.461500","63,867.758213","13,215.114864","2,185.159967",640.809375,"15,786.205277","32,040.468731",1.154717,60.045271,734.000000,weekly_net,next_open,recent_1y_weekly_walk_forward,260,52,1,52


**长期 Walk-forward 摘要**

,evaluation_name,rolling_eval_dates,first_eval_date,last_eval_date,train_weeks,valid_weeks,step_weeks,eval_count,start_date,end_date,topk,backtest_mode,execution_price,execution_lag_steps,trade_unit,stop_loss_mode
0,long_walk_forward,203,2022-03-11,2026-02-27,260,52,1,0,2016-01-01,2026-03-27,10,weekly_net,next_open,1,100,off


,recipe,evaluation_dates,coverage_mean,rank_ic_mean,rank_ic_std,rank_ic_ir,ic_mean,ic_std,topk_mean_return_4w,topk_mean_excess_return_4w,topk_hit_rate,universe_mean_return_4w,used_feature_count,evaluation_name,train_weeks,valid_weeks,step_weeks,eval_count
0,long_walk_forward,203,298.157635,0.033185,0.134526,0.246679,0.019006,0.112024,0.010173,0.001348,0.486700,0.005215,36,long_walk_forward,260,52,1,0


,evaluation_name,strategy_total_return,strategy_cagr,strategy_max_drawdown,strategy_weekly_sharpe,strategy_total_cost,strategy_commission,strategy_exchange_fee,strategy_transfer_fee,strategy_stamp_duty,strategy_impact_cost,strategy_gross_trade_value,strategy_turnover_ratio_total,strategy_trade_count,backtest_mode,execution_price
0,long_walk_forward,0.321590,0.072815,-0.286940,0.415699,"218,800.523623","45,548.174562","7,448.133508","2,184.203375","54,409.843436","109,210.168742","218,420,337.483500",218.138042,"2,795.000000",weekly_net,next_open


,recipe,slice,evaluation_dates,coverage_mean,rank_ic_mean,rank_ic_std,rank_ic_ir,ic_mean,ic_std,topk_mean_return_4w,topk_mean_excess_return_4w,topk_hit_rate,universe_mean_return_4w,used_feature_count,strategy_total_return,strategy_cagr,strategy_max_drawdown,strategy_weekly_sharpe,strategy_gross_trade_value,strategy_total_cost,strategy_commission,strategy_exchange_fee,strategy_transfer_fee,strategy_stamp_duty,strategy_impact_cost,strategy_turnover_ratio_mean,strategy_turnover_ratio_total,strategy_trade_count,backtest_mode,execution_price,evaluation_name,train_weeks,valid_weeks,step_weeks,eval_count
0,long_walk_forward,csi300,203,298.157635,0.033185,0.134526,0.246679,0.019006,0.112024,0.010173,0.001348,0.486700,0.005215,36,0.321590,0.072815,-0.286940,0.415699,"218,420,337.483500","218,800.523623","45,548.174562","7,448.133508","2,184.203375","54,409.843436","109,210.168742",1.074572,218.138042,"2,795.000000",weekly_net,next_open,long_walk_forward,260,52,1,0


## 9. 分析与可视化

这一节聚合输出特征重要性、快速评估与长期 walk-forward 的 RankIC/TopK 序列、净值曲线、切片表现和最新预测表。

In [9]:
display(Markdown("**快速滚动评估摘要**"))
display(rolling_summary)
display(rolling_backtest_metric_frame)
if LONG_WALK_FORWARD_ENABLED and not walk_forward_summary.empty:
    display(Markdown("**长期 Walk-forward 摘要**"))
    display(walk_forward_summary)
    display(walk_forward_backtest_metric_frame)

overview_rows = [
    {
        "evaluation_name": "recent_1y_weekly_walk_forward",
        "rank_ic_ir": _format_metric(rolling_summary.iloc[0]["rank_ic_ir"]) if not rolling_summary.empty else "NA",
        "topk_mean_excess_return_4w": _format_metric(rolling_summary.iloc[0]["topk_mean_excess_return_4w"]) if not rolling_summary.empty else "NA",
        "strategy_total_return": _format_metric(backtest_metrics.get("strategy_total_return")),
        "strategy_max_drawdown": _format_metric(backtest_metrics.get("strategy_max_drawdown")),
        "strategy_total_cost": _format_metric(backtest_metrics.get("strategy_total_cost")),
        "strategy_turnover_ratio_total": _format_metric(backtest_metrics.get("strategy_turnover_ratio_total")),
        "evaluation_dates": len(rolling_eval_dates),
    }
]
if LONG_WALK_FORWARD_ENABLED and not walk_forward_summary.empty:
    overview_rows.append(
        {
            "evaluation_name": "long_walk_forward",
            "rank_ic_ir": _format_metric(walk_forward_summary.iloc[0]["rank_ic_ir"]),
            "topk_mean_excess_return_4w": _format_metric(walk_forward_summary.iloc[0]["topk_mean_excess_return_4w"]),
            "strategy_total_return": _format_metric(walk_forward_backtest_metrics.get("strategy_total_return")),
            "strategy_max_drawdown": _format_metric(walk_forward_backtest_metrics.get("strategy_max_drawdown")),
            "strategy_total_cost": _format_metric(walk_forward_backtest_metrics.get("strategy_total_cost")),
            "strategy_turnover_ratio_total": _format_metric(walk_forward_backtest_metrics.get("strategy_turnover_ratio_total")),
            "evaluation_dates": len(walk_forward_eval_dates),
        }
    )
display(Markdown("**窗口结果对比**"))
display(pd.DataFrame(overview_rows))

cost_frames = [rolling_backtest_metric_frame.copy()] if not rolling_backtest_metric_frame.empty else []
if LONG_WALK_FORWARD_ENABLED and not walk_forward_backtest_metric_frame.empty:
    cost_frames.append(walk_forward_backtest_metric_frame.copy())
if cost_frames:
    cost_compare = pd.concat(cost_frames, ignore_index=True)
    display(Markdown("**净收益回测成本拆解**"))
    display(
        cost_compare[
            [
                "evaluation_name",
                "strategy_total_cost",
                "strategy_commission",
                "strategy_exchange_fee",
                "strategy_transfer_fee",
                "strategy_stamp_duty",
                "strategy_impact_cost",
                "strategy_gross_trade_value",
                "strategy_turnover_ratio_total",
                "strategy_trade_count",
                "backtest_mode",
                "execution_price",
            ]
        ]
    )

importance_fig = px.bar(
    feature_importance_frame.head(20).sort_values("importance", ascending=True),
    x="importance",
    y="feature",
    orientation="h",
    title="特征重要性 Top20",
)
importance_fig.update_layout(height=600)
importance_fig.show()

detail_frames = [rolling_details.copy()]
if LONG_WALK_FORWARD_ENABLED and not walk_forward_details.empty:
    detail_frames.append(walk_forward_details.copy())
detail_plot = pd.concat(detail_frames, ignore_index=True).melt(
    id_vars=["feature_date", "recipe", "evaluation_name"],
    value_vars=["rank_ic", "topk_mean_excess_return_4w"],
    var_name="metric",
    value_name="value",
)
detail_plot["series"] = detail_plot["evaluation_name"].astype(str) + " | " + detail_plot["metric"].astype(str)
detail_fig = px.line(
    detail_plot,
    x="feature_date",
    y="value",
    color="series",
    title="快速评估与长期 Walk-forward 的 RankIC / TopK 超额收益时间序列",
)
detail_fig.update_layout(height=520)
detail_fig.show()

equity_plot_frames = []
if not equity_curve_frame.empty:
    equity_plot_frames.append(equity_curve_frame.copy())
if not slice_equity_curve_frame.empty:
    equity_plot_frames.append(slice_equity_curve_frame.copy())
if LONG_WALK_FORWARD_ENABLED and not walk_forward_equity_curve_frame.empty:
    equity_plot_frames.append(walk_forward_equity_curve_frame.copy())
if LONG_WALK_FORWARD_ENABLED and not walk_forward_slice_equity_curve_frame.empty:
    equity_plot_frames.append(walk_forward_slice_equity_curve_frame.copy())

if equity_plot_frames:
    equity_plot = pd.concat(equity_plot_frames, ignore_index=True)
    equity_plot["legend"] = equity_plot["evaluation_name"].astype(str) + " | " + equity_plot["series"].astype(str)
    equity_fig = px.line(
        equity_plot,
        x="datetime",
        y="total_value",
        color="legend",
        title="净值曲线：快速评估 vs 长期 Walk-forward",
    )
    equity_fig.update_layout(height=520)
    equity_fig.show()

rolling_monthly_return_heatmap = _build_monthly_return_heatmap_frame(equity_curve_frame)
rolling_annual_return_heatmap = _build_annual_return_heatmap_frame(equity_curve_frame)
display(Markdown("**快速滚动评估收益率热力图**"))
_render_return_heatmap(
    rolling_monthly_return_heatmap,
    title="快速滚动评估月度收益率热力图",
    x_label="Month",
    y_label="Year",
)
_render_return_heatmap(
    rolling_annual_return_heatmap,
    title="快速滚动评估年度收益率热力图",
    x_label="Year",
    y_label="Series",
)

if LONG_WALK_FORWARD_ENABLED and not walk_forward_equity_curve_frame.empty:
    walk_forward_monthly_return_heatmap = _build_monthly_return_heatmap_frame(walk_forward_equity_curve_frame)
    walk_forward_annual_return_heatmap = _build_annual_return_heatmap_frame(walk_forward_equity_curve_frame)
    display(Markdown("**长期 Walk-forward 收益率热力图**"))
    _render_return_heatmap(
        walk_forward_monthly_return_heatmap,
        title="长期 Walk-forward 月度收益率热力图",
        x_label="Month",
        y_label="Year",
    )
    _render_return_heatmap(
        walk_forward_annual_return_heatmap,
        title="长期 Walk-forward 年度收益率热力图",
        x_label="Year",
        y_label="Series",
    )
else:
    walk_forward_monthly_return_heatmap = pd.DataFrame()
    walk_forward_annual_return_heatmap = pd.DataFrame()
    if LONG_WALK_FORWARD_ENABLED:
        display(Markdown("**长期 Walk-forward 收益率热力图**：净值曲线为空，已跳过。"))
    else:
        display(Markdown("**长期 Walk-forward 收益率热力图**：当前未启用长期 Walk-forward。"))

slice_summary_frames = []
if not slice_summary_frame.empty:
    slice_summary_frames.append(slice_summary_frame.copy())
if LONG_WALK_FORWARD_ENABLED and not walk_forward_slice_summary_frame.empty:
    slice_summary_frames.append(walk_forward_slice_summary_frame.copy())

if slice_summary_frames:
    combined_slice_summary = pd.concat(slice_summary_frames, ignore_index=True)
    slice_rank_fig = px.bar(
        combined_slice_summary,
        x="slice",
        y="rank_ic_ir",
        color="evaluation_name",
        barmode="group",
        title="切片 RankIC_IR 对比",
    )
    slice_rank_fig.update_layout(height=420)
    slice_rank_fig.show()
else:
    display(Markdown("**切片图表**：当前 universe 无可展示的 CSI300 / CSI500 切片。"))

display(Markdown(f"**最新预测日期**：`{pd.Timestamp(latest_feature_date).date()}`"))
display(
    latest_prediction_with_labels[
        [
            column
            for column in [
                "instrument",
                "score",
                "future_return_4w",
                LABEL_COLUMN,
                "l1_name",
                "in_csi300",
                "in_csi500",
            ]
            if column in latest_prediction_with_labels.columns
        ]
    ].head(20)
)


**快速滚动评估摘要**

,recipe,evaluation_dates,coverage_mean,rank_ic_mean,rank_ic_std,rank_ic_ir,ic_mean,ic_std,topk_mean_return_4w,topk_mean_excess_return_4w,topk_hit_rate,universe_mean_return_4w,used_feature_count,evaluation_name,train_weeks,valid_weeks,step_weeks,eval_count
0,recent_1y_weekly_walk_forward,52,298.903846,0.031613,0.153285,0.206237,-0.003055,0.129674,0.024613,0.008313,0.561538,0.015245,36,recent_1y_weekly_walk_forward,260,52,1,52


,evaluation_name,strategy_total_return,strategy_cagr,strategy_max_drawdown,strategy_weekly_sharpe,strategy_total_cost,strategy_commission,strategy_exchange_fee,strategy_transfer_fee,strategy_stamp_duty,strategy_impact_cost,strategy_gross_trade_value,strategy_turnover_ratio_total,strategy_trade_count,backtest_mode,execution_price
0,recent_1y_weekly_walk_forward,0.283559,0.284660,-0.085461,1.441517,"63,867.758213","13,215.114864","2,185.159967",640.809375,"15,786.205277","32,040.468731","64,080,937.461500",60.045271,734.000000,weekly_net,next_open


**长期 Walk-forward 摘要**

,recipe,evaluation_dates,coverage_mean,rank_ic_mean,rank_ic_std,rank_ic_ir,ic_mean,ic_std,topk_mean_return_4w,topk_mean_excess_return_4w,topk_hit_rate,universe_mean_return_4w,used_feature_count,evaluation_name,train_weeks,valid_weeks,step_weeks,eval_count
0,long_walk_forward,203,298.157635,0.033185,0.134526,0.246679,0.019006,0.112024,0.010173,0.001348,0.486700,0.005215,36,long_walk_forward,260,52,1,0


...

,evaluation_name,strategy_total_cost,strategy_commission,strategy_exchange_fee,strategy_transfer_fee,strategy_stamp_duty,strategy_impact_cost,strategy_gross_trade_value,strategy_turnover_ratio_total,strategy_trade_count,backtest_mode,execution_price
0,recent_1y_weekly_walk_forward,"63,867.758213","13,215.114864","2,185.159967",640.809375,"15,786.205277","32,040.468731","64,080,937.461500",60.045271,734.000000,weekly_net,next_open
1,long_walk_forward,"218,800.523623","45,548.174562","7,448.133508","2,184.203375","54,409.843436","109,210.168742","218,420,337.483500",218.138042,"2,795.000000",weekly_net,next_open


**快速滚动评估收益率热力图**

**长期 Walk-forward 收益率热力图**

**最新预测日期**：`2026-03-27`

,instrument,score,future_return_4w,label_excess_return_4w,l1_name,in_csi300
0,688506.SH,0.001827,NaN,NaN,医药生物,True
1,601100.SH,0.001318,NaN,NaN,机械设备,True
2,300033.SZ,0.001200,NaN,NaN,计算机,True
3,601888.SH,-0.000769,NaN,NaN,商贸零售,True
4,603893.SH,-0.000769,NaN,NaN,电子,True
5,688012.SH,-0.000769,NaN,NaN,电子,True
6,601225.SH,-0.000852,NaN,NaN,煤炭,True
7,600233.SH,-0.001468,NaN,NaN,交通运输,True
8,600660.SH,-0.003067,NaN,NaN,汽车,True
9,300760.SZ,-0.003773,NaN,NaN,医药生物,True


## 10. 可选发布与产物落盘

默认只把 notebook 研究产物写到 `OUTPUT_DIR`；只有在 `RUN_PUBLISH=True` 时才保存模型文件并发布线上快照。

In [ ]:
output_dir_path = (PROJECT_ROOT / OUTPUT_DIR).resolve()
output_dir_path.mkdir(parents=True, exist_ok=True)

feature_stats.to_csv(output_dir_path / "feature_prefilter.csv", index=False)
corr_marks.to_csv(output_dir_path / "feature_corr_candidates.csv", index=False)
feature_importance_frame.to_csv(output_dir_path / "feature_importance.csv", index=False)
latest_prediction_with_labels.to_csv(output_dir_path / "latest_prediction.csv", index=False)
rolling_summary.to_csv(output_dir_path / "rolling_summary.csv", index=False)
rolling_backtest_metric_frame.to_csv(output_dir_path / "rolling_backtest_metrics.csv", index=False)
rolling_details.to_csv(output_dir_path / "rolling_details.csv", index=False)
rolling_predictions.to_csv(output_dir_path / "rolling_predictions.csv", index=False)
if not rolling_monthly_return_heatmap.empty:
    rolling_monthly_return_heatmap.to_csv(output_dir_path / "rolling_monthly_return_heatmap.csv")
if not rolling_annual_return_heatmap.empty:
    rolling_annual_return_heatmap.to_csv(output_dir_path / "rolling_annual_return_heatmap.csv")
if not slice_summary_frame.empty:
    slice_summary_frame.to_csv(output_dir_path / "slice_summary.csv", index=False)
if not equity_curve_frame.empty:
    equity_curve_frame.to_csv(output_dir_path / "equity_curve.csv", index=False)
if not slice_equity_curve_frame.empty:
    slice_equity_curve_frame.to_csv(output_dir_path / "slice_equity_curve.csv", index=False)
if LONG_WALK_FORWARD_ENABLED and not walk_forward_summary.empty:
    walk_forward_summary.to_csv(output_dir_path / "walk_forward_summary.csv", index=False)
if LONG_WALK_FORWARD_ENABLED and not walk_forward_backtest_metric_frame.empty:
    walk_forward_backtest_metric_frame.to_csv(output_dir_path / "walk_forward_backtest_metrics.csv", index=False)
if LONG_WALK_FORWARD_ENABLED and not walk_forward_details.empty:
    walk_forward_details.to_csv(output_dir_path / "walk_forward_details.csv", index=False)
if LONG_WALK_FORWARD_ENABLED and not walk_forward_predictions.empty:
    walk_forward_predictions.to_csv(output_dir_path / "walk_forward_predictions.csv", index=False)
if LONG_WALK_FORWARD_ENABLED and not walk_forward_monthly_return_heatmap.empty:
    walk_forward_monthly_return_heatmap.to_csv(output_dir_path / "walk_forward_monthly_return_heatmap.csv")
if LONG_WALK_FORWARD_ENABLED and not walk_forward_annual_return_heatmap.empty:
    walk_forward_annual_return_heatmap.to_csv(output_dir_path / "walk_forward_annual_return_heatmap.csv")
if LONG_WALK_FORWARD_ENABLED and not walk_forward_equity_curve_frame.empty:
    walk_forward_equity_curve_frame.to_csv(output_dir_path / "walk_forward_equity_curve.csv", index=False)
if LONG_WALK_FORWARD_ENABLED and not walk_forward_slice_summary_frame.empty:
    walk_forward_slice_summary_frame.to_csv(output_dir_path / "walk_forward_slice_summary.csv", index=False)
if LONG_WALK_FORWARD_ENABLED and not walk_forward_slice_equity_curve_frame.empty:
    walk_forward_slice_equity_curve_frame.to_csv(output_dir_path / "walk_forward_slice_equity_curve.csv", index=False)

publish_result = {
    "published": False,
    "reason": "run_publish_disabled",
    "candidate_model_id": CANDIDATE_MODEL_ID,
    "feature_date": str(pd.Timestamp(latest_feature_date).date()),
    "artifact_dir": str(output_dir_path),
}

if RUN_PUBLISH:
    artifacts_root = (PROJECT_ROOT / "data" / "qlib_artifacts").resolve()
    model_dir = artifacts_root / CANDIDATE_MODEL_ID
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / "lgb_model.txt"
    model.model.save_model(str(model_path))
    snapshot_path = publish_score_snapshot(
        score_frame=score_snapshot_frame,
        model_id=CANDIDATE_MODEL_ID,
        feature_date=str(pd.Timestamp(latest_feature_date).date()),
        extra_manifest={
            "label_column": LABEL_COLUMN,
            "feature_columns": list(used_feature_columns),
            "selected_feature_groups": list(ACTIVE_FEATURE_GROUPS),
            "industry_normalization": INDUSTRY_NORMALIZATION,
            "tuned_params": MODEL_PARAMS,
            "evaluation_scope": UNIVERSE_PROFILE,
            "workflow_style": "explicit_workflow_by_code",
            "rolling_eval_count": len(rolling_eval_dates),
            "backtest_mode": BACKTEST_MODE,
            "execution_price": EXECUTION_PRICE,
            "execution_lag_steps": EXECUTION_LAG_STEPS,
            "backtest_config": {
                "broker_commission_rate": BROKER_COMMISSION_RATE,
                "exchange_fee_rate": EXCHANGE_FEE_RATE,
                "transfer_fee_rate": TRANSFER_FEE_RATE,
                "stamp_duty_sell_rate": STAMP_DUTY_SELL_RATE,
                "impact_cost_rate": IMPACT_COST_RATE,
                "min_commission": MIN_COMMISSION,
                "trade_unit": TRADE_UNIT,
                "stop_loss_mode": STOP_LOSS_MODE,
                "stop_loss_threshold": STOP_LOSS_THRESHOLD,
            },
            "qlib_exchange_cost_preview": backtest_config.qlib_exchange_kwargs(deal_price="$open"),
            "long_walk_forward_enabled": LONG_WALK_FORWARD_ENABLED,
            "long_walk_forward_eval_count": len(walk_forward_eval_dates),
            "long_walk_forward_start_date": LONG_WALK_FORWARD_START_DATE,
            "long_walk_forward_end_date": LONG_WALK_FORWARD_END_DATE,
            "long_walk_forward_step_weeks": LONG_WALK_FORWARD_STEP_WEEKS,
            "model_path": model_path.name,
        },
    )
    publish_result = {
        "published": True,
        "candidate_model_id": CANDIDATE_MODEL_ID,
        "feature_date": str(pd.Timestamp(latest_feature_date).date()),
        "model_path": str(model_path),
        "snapshot_path": str(snapshot_path),
        "artifact_dir": str(output_dir_path),
    }

display(
    pd.DataFrame(
        [
            {
                "RUN_PUBLISH": RUN_PUBLISH,
                "candidate_model_id": CANDIDATE_MODEL_ID,
                "artifact_dir": str(output_dir_path),
                "latest_feature_date": str(pd.Timestamp(latest_feature_date).date()),
                "prediction_count": len(score_snapshot_frame),
                "rolling_eval_count": len(rolling_eval_dates),
                "backtest_mode": BACKTEST_MODE,
                "execution_price": EXECUTION_PRICE,
                "strategy_total_cost": backtest_metrics.get("strategy_total_cost", 0.0),
                "long_walk_forward_enabled": LONG_WALK_FORWARD_ENABLED,
                "long_walk_forward_eval_count": len(walk_forward_eval_dates),
            }
        ]
    )
)
display(pd.DataFrame([publish_result]))
